<a href="https://colab.research.google.com/github/6pb4wnww4g-beep/Jason/blob/main/Jason_Market_xG_All_Fixtures_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- JASON MARKET xG ALL FIXTURES v1 REBUILT ---
# Formål:
# - Gjenbygger missing link: Jason_Market_xG_All_Fixtures_v1.py
# - Leser bookmaker-detaljer fra OddsAPI
# - Estimerer Team xG / Opp xG / CS% / Win% / Draw% / Loss%
# - Skriver Market_xG_All_Fixtures_v1
#
# Leser først:
#   OddsAPI_GW1_5_Bookmaker_Detail
#
# Fallback:
#   OddsAPI_Bookmaker_Detail
#
# Skriver:
#   Market_xG_All_Fixtures_v1
#   Market_xG_Parse_Check_v1
#
# Neste script i kjeden:
#   JASON MARKET TEAM STRENGTH v2.1 FIXED

import time
import math
import numpy as np
import pandas as pd
from google.colab import auth
import gspread
from google.auth import default

SPREADSHEET_NAME = "Jason Development"

INPUT_SHEETS = [
    "OddsAPI_GW1_5_Bookmaker_Detail",
    "OddsAPI_Bookmaker_Detail",
]

OUT_SHEET = "Market_xG_All_Fixtures_v1"
CHECK_SHEET = "Market_xG_Parse_Check_v1"

DEFAULT_TOTAL_GOALS = 2.65


auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.open(SPREADSHEET_NAME)


def parse_float(v, default=np.nan):
    if v is None:
        return default

    s = str(v).strip()
    if s == "":
        return default

    s = s.replace(" ", "")

    if "," in s and "." not in s:
        s = s.replace(",", ".")
    elif "," in s and "." in s:
        s = s.replace(",", "")

    try:
        return float(s)
    except Exception:
        return default


def read_first_available_sheet(sheet_names):
    last_error = None

    for sheet_name in sheet_names:
        try:
            ws = sh.worksheet(sheet_name)
            values = ws.get_all_values()

            if not values or len(values) < 2:
                continue

            headers = values[0]
            rows = values[1:]
            df = pd.DataFrame(rows, columns=headers)

            print(f"Leser input: {sheet_name} ({len(df)} rader)")
            return sheet_name, df

        except Exception as e:
            last_error = e
            continue

    raise ValueError(f"Fant ingen gyldige input-ark: {sheet_names}. Siste feil: {last_error}")


def update_worksheet(sheet_name, dataframe):
    print(f"Oppdaterer {sheet_name}...")
    df_clean = dataframe.replace([np.inf, -np.inf], np.nan).fillna("")

    try:
        ws = sh.worksheet(sheet_name)
        ws.clear()
    except Exception:
        ws = sh.add_worksheet(title=sheet_name, rows="2000", cols="80")

    ws.resize(
        rows=max(200, len(df_clean) + 20),
        cols=max(25, len(df_clean.columns) + 5)
    )

    ws.update([df_clean.columns.tolist()] + df_clean.values.tolist())
    time.sleep(2)


def avg_no_vig_probs(prices):
    implied = {}

    for outcome, odds_list in prices.items():
        clean = [parse_float(x) for x in odds_list]
        clean = [x for x in clean if not pd.isna(x) and x > 1.0]

        if clean:
            avg_odds = sum(clean) / len(clean)
            implied[outcome] = 1.0 / avg_odds

    total = sum(implied.values())

    if total <= 0:
        return {}

    return {k: v / total for k, v in implied.items()}


def poisson_pmf(lam, k):
    return math.exp(-lam) * (lam ** k) / math.factorial(k)


def poisson_match_probs(home_xg, away_xg, max_goals=10):
    home_win = 0.0
    draw = 0.0
    away_win = 0.0
    total_under_25 = 0.0

    home_zero = poisson_pmf(home_xg, 0)
    away_zero = poisson_pmf(away_xg, 0)

    home_cs = away_zero
    away_cs = home_zero

    for h in range(max_goals + 1):
        ph = poisson_pmf(home_xg, h)

        for a in range(max_goals + 1):
            pa = poisson_pmf(away_xg, a)
            p = ph * pa

            if h > a:
                home_win += p
            elif h == a:
                draw += p
            else:
                away_win += p

            if h + a <= 2:
                total_under_25 += p

    return {
        "home_win": home_win,
        "draw": draw,
        "away_win": away_win,
        "home_cs": home_cs,
        "away_cs": away_cs,
        "under_25": total_under_25,
        "over_25": 1.0 - total_under_25,
    }


def expected_total_from_over25(over_prob):
    if over_prob is None or pd.isna(over_prob):
        return DEFAULT_TOTAL_GOALS

    target = max(0.05, min(0.95, float(over_prob)))

    best_lam = DEFAULT_TOTAL_GOALS
    best_err = 999

    for lam in np.arange(0.60, 5.51, 0.01):
        p_under_25 = sum(poisson_pmf(lam, k) for k in range(0, 3))
        p_over_25 = 1.0 - p_under_25
        err = abs(p_over_25 - target)

        if err < best_err:
            best_err = err
            best_lam = lam

    return float(best_lam)


def fit_xg_from_market(home_win_p, draw_p, away_win_p, total_goals):
    total_goals = max(0.6, min(5.5, float(total_goals)))

    best = {
        "home_xg": total_goals / 2,
        "away_xg": total_goals / 2,
        "err": 999,
        "probs": None,
    }

    for home_share in np.arange(0.15, 0.86, 0.0025):
        hxg = total_goals * home_share
        axg = total_goals - hxg

        probs = poisson_match_probs(hxg, axg)

        err = (
            abs(probs["home_win"] - home_win_p) +
            abs(probs["draw"] - draw_p) +
            abs(probs["away_win"] - away_win_p)
        )

        if err < best["err"]:
            best = {
                "home_xg": hxg,
                "away_xg": axg,
                "err": err,
                "probs": probs,
            }

    return best


def build_market_xg(detail_df):
    required = [
        "match_id",
        "commence_time",
        "home_team",
        "away_team",
        "bookmaker",
        "market",
        "outcome_name",
        "price",
    ]

    missing = [c for c in required if c not in detail_df.columns]
    if missing:
        raise ValueError(f"Input mangler kolonner: {missing}")

    if "point" not in detail_df.columns:
        detail_df["point"] = ""

    rows = []
    check_rows = []

    grouped = detail_df.groupby(["match_id", "commence_time", "home_team", "away_team"], dropna=False)

    for (match_id, kickoff, home, away), g in grouped:
        home = str(home).strip()
        away = str(away).strip()

        h2h = g[g["market"].astype(str).str.lower().eq("h2h")].copy()
        totals = g[g["market"].astype(str).str.lower().eq("totals")].copy()

        h2h_prices = {
            "home": [],
            "draw": [],
            "away": [],
        }

        for _, r in h2h.iterrows():
            outcome = str(r.get("outcome_name", "")).strip()
            price = r.get("price", "")

            if outcome == home:
                h2h_prices["home"].append(price)
            elif outcome == away:
                h2h_prices["away"].append(price)
            elif outcome.lower() == "draw":
                h2h_prices["draw"].append(price)

        h2h_probs = avg_no_vig_probs(h2h_prices)

        home_p = h2h_probs.get("home", np.nan)
        draw_p = h2h_probs.get("draw", np.nan)
        away_p = h2h_probs.get("away", np.nan)

        over_prices = []
        under_prices = []
        selected_point = np.nan

        if not totals.empty:
            totals["point_num"] = totals["point"].apply(parse_float)
            points = sorted([p for p in totals["point_num"].dropna().unique().tolist()])

            if points:
                selected_point = min(points, key=lambda x: abs(x - 2.5))
                tsel = totals[totals["point_num"].eq(selected_point)]

                for _, r in tsel.iterrows():
                    outcome = str(r.get("outcome_name", "")).strip().lower()
                    price = r.get("price", "")

                    if outcome == "over":
                        over_prices.append(price)
                    elif outcome == "under":
                        under_prices.append(price)

        total_probs = avg_no_vig_probs({
            "over": over_prices,
            "under": under_prices,
        })

        over_p = total_probs.get("over", np.nan)
        total_goals = expected_total_from_over25(over_p)

        if any(pd.isna(x) for x in [home_p, draw_p, away_p]):
            check_rows.append({
                "match_id": match_id,
                "home_team": home,
                "away_team": away,
                "status": "MISSING_H2H",
                "h2h_rows": len(h2h),
                "totals_rows": len(totals),
            })
            continue

        fit = fit_xg_from_market(home_p, draw_p, away_p, total_goals)
        hxg = fit["home_xg"]
        axg = fit["away_xg"]
        probs = fit["probs"] or poisson_match_probs(hxg, axg)

        rows.append({
            "Kickoff": kickoff,
            "Team": home,
            "Opponent": away,
            "H/A": "H",
            "Team xG": round(hxg, 4),
            "Opp xG": round(axg, 4),
            "CS%": round(probs["home_cs"] * 100, 4),
            "Win%": round(probs["home_win"] * 100, 4),
            "Draw%": round(probs["draw"] * 100, 4),
            "Loss%": round(probs["away_win"] * 100, 4),
            "Total xG": round(hxg + axg, 4),
            "Fit Error": round(fit["err"], 4),
            "Match ID": match_id,
        })

        rows.append({
            "Kickoff": kickoff,
            "Team": away,
            "Opponent": home,
            "H/A": "A",
            "Team xG": round(axg, 4),
            "Opp xG": round(hxg, 4),
            "CS%": round(probs["away_cs"] * 100, 4),
            "Win%": round(probs["away_win"] * 100, 4),
            "Draw%": round(probs["draw"] * 100, 4),
            "Loss%": round(probs["home_win"] * 100, 4),
            "Total xG": round(hxg + axg, 4),
            "Fit Error": round(fit["err"], 4),
            "Match ID": match_id,
        })

        check_rows.append({
            "match_id": match_id,
            "kickoff": kickoff,
            "home_team": home,
            "away_team": away,
            "status": "OK",
            "h2h_home_p": round(home_p, 4),
            "h2h_draw_p": round(draw_p, 4),
            "h2h_away_p": round(away_p, 4),
            "selected_total_line": selected_point,
            "over_p": round(over_p, 4) if not pd.isna(over_p) else "",
            "total_goals_est": round(total_goals, 4),
            "home_xg": round(hxg, 4),
            "away_xg": round(axg, 4),
            "fit_error": round(fit["err"], 4),
            "h2h_rows": len(h2h),
            "totals_over_rows": len(over_prices),
            "totals_under_rows": len(under_prices),
        })

    out_df = pd.DataFrame(rows)
    check_df = pd.DataFrame(check_rows)

    if not out_df.empty:
        out_df = out_df.sort_values(["Kickoff", "Team"]).reset_index(drop=True)

    if not check_df.empty:
        check_df = check_df.sort_values(["kickoff", "home_team"], na_position="last").reset_index(drop=True)

    return out_df, check_df


input_sheet, detail_df = read_first_available_sheet(INPUT_SHEETS)
out_df, check_df = build_market_xg(detail_df)

update_worksheet(OUT_SHEET, out_df)
update_worksheet(CHECK_SHEET, check_df)

print("Ferdig.")
print(f"Input brukt: {input_sheet}")
print(f"Skrev: {OUT_SHEET} ({len(out_df)} rader)")
print(f"Skrev: {CHECK_SHEET} ({len(check_df)} rader)")
print("")
print("Neste steg:")
print("Kjør JASON MARKET TEAM STRENGTH v2.1 FIXED.")